In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import math
import os
from pathlib import Path
import celltypist
from celltypist import models

sc.settings.set_figure_params(dpi=300, frameon=False, facecolor='white')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.sans-serif'] = 'Arial'

# Path configuration (Masking absolute paths)
FILE_PATH = Path(os.getenv("SC_INTEGRATED_H5AD", "./data/processed/adata_integrated_cleaned.h5ad"))
MODEL_DIR = Path(os.getenv("CELLTYPIST_MODEL_DIR", "./models/celltypist"))
SAVE_DIR_EVIDENCE = Path(os.getenv("SC_EVIDENCE_OUT", "./results/final_annotation/Evidence_Plots_Refined"))

os.makedirs(SAVE_DIR_EVIDENCE, exist_ok=True)

# Load data
adata = sc.read_h5ad(FILE_PATH)
adata.obs_names_make_unique()

# 1. CellTypist Annotation
model_file = "Developing_Human_Brain.pkl"
model_path = MODEL_DIR / model_file

if model_path.exists():
    predictions = celltypist.annotate(
        adata, 
        model=str(model_path), 
        majority_voting=True,
        over_clustering='leiden_res_0.6'
    )
    result_df = predictions.predicted_labels
    adata.obs['celltypist_prediction'] = result_df['predicted_labels']
    adata.obs['celltypist_majority'] = result_df['majority_voting']
    print("CellTypist Annotation Done")
    print(adata.obs[['leiden_res_0.6', 'celltypist_majority']].head())
else:
    print(f"Error: Model file not found at {model_path}")

# 2. Evidence Plots Setup
default_vmax = 5
default_size = 1.3

special_configs = {
    'TFAP2B': {'vmax': 3.0, 'size': 1.2}, 
    'PAX3':   {'vmax': 4.0, 'size': 1.2}, 
    'LMO4':   {'vmax': 4.0, 'size': 1.2}
}

custom_cmap = mcolors.LinearSegmentedColormap.from_list("gray_blue_specific", ["#e0e0e0", "#d1d1d1", "#2166ac", "#053061"])

evidence_markers = {
    'RG_NPC': ['DACH1', 'COL4A6', 'GPC3', 'PAX6', 'SOX2', 'VIM'],
    'Cycling': ['TOP2A', 'MKI67', 'CENPF'],                      
    'Immature_Neurons': ['STMN2', 'DCX', 'RBFOX1'],
    'Neural_Crest': ['TFAP2B', 'PAX3', 'LMO4'],
    'Fibroblast': ['KRT18', 'KRT19', 'S100A11']
}

# 3. Dynamic Layout Plotting Loop
for module_name, genes in evidence_markers.items():
    available_genes = [g for g in genes if g in adata.var_names]
    if not available_genes:
        print(f"Warning: Skipping module {module_name} (no genes found)")
        continue

    print(f"Processing module markers: {module_name}")
    
    n_genes = len(genes)
    n_cols = 3
    n_rows = math.ceil(n_genes / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 5))
    if n_rows * n_cols > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    for i in range(len(axes)):
        ax = axes[i]
        if i < len(genes):
            gene = genes[i]
            if gene not in adata.var_names:
                ax.text(0.5, 0.5, f"{gene}\nNot Found", ha='center', va='center')
                ax.axis('off')
                continue
            
            cfg = special_configs.get(gene, {'vmax': default_vmax, 'size': default_size})
            
            sc.pl.umap(
                adata, 
                color=gene, 
                cmap=custom_cmap, 
                vmax=cfg['vmax'],
                size=10, 
                sort_order=True, 
                frameon=False, 
                ax=ax, 
                show=False
            )

            for child in ax.get_children():
                if isinstance(child, plt.matplotlib.collections.PathCollection):
                    child.set_rasterized(True)
            
            ax.set_title(gene, fontsize=14, fontweight='bold', style='italic')
        else:
            ax.axis('off')

    plt.tight_layout()
    save_path = SAVE_DIR_EVIDENCE / f"Evidence_{module_name}_Specific.pdf"
    plt.savefig(save_path, bbox_inches='tight', transparent=True)
    plt.show()

# 4. IPC Verification Map
sc.pl.umap(
    adata, 
    color=['PAX6', 'EOMES', 'NEUROD1'], 
    cmap=custom_cmap, 
    vmax='p99', 
    size=1.5, 
    ncols=3, 
    title=['RG Marker (PAX6)', 'IPC Gold Standard (EOMES)', 'Early Neuron (NEUROD1)']
)